In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Github Connection
!git clone https://github.com/laureneproctor/Olympiad-AI.git
%cd Olympiad-AI

In [ ]:
import random
import re
from pathlib import Path

import torch
from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer

try:
    from peft import AutoPeftModelForCausalLM
except Exception:
    AutoPeftModelForCausalLM = None


FINAL_RE = re.compile(r"FINAL_ANSWER:\\s*([+-]?\\d+)\\b")
LAST_INT_RE = re.compile(r"\\b([+-]?\\d+)\\b")


def get_torch_dtype():
    if torch.cuda.is_available():
        if torch.cuda.is_bf16_supported():
            return torch.bfloat16
        return torch.float16
    return torch.float32


def extract_model_answer(text):
    if text is None:
        return None
    match = FINAL_RE.findall(text)
    if match:
        return str(int(match[-1]))
    fallback = LAST_INT_RE.findall(text)
    if fallback:
        return str(int(fallback[-1]))
    return None


def load_demo_model(checkpoint_path, tokenizer_path=None, trust_remote_code=True):
    tokenizer_candidates = [checkpoint_path]
    if tokenizer_path:
        tokenizer_candidates.insert(0, tokenizer_path)

    tok = None
    last_error = None
    for candidate in tokenizer_candidates:
        try:
            tok = AutoTokenizer.from_pretrained(candidate, use_fast=True, trust_remote_code=trust_remote_code)
            break
        except Exception as e:
            last_error = e

    if tok is None:
        raise ValueError(f"Could not load tokenizer. Last error: {last_error}")

    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"

    model_kwargs = {
        "device_map": "auto",
        "torch_dtype": get_torch_dtype(),
        "trust_remote_code": trust_remote_code,
    }

    model = None
    if AutoPeftModelForCausalLM is not None:
        try:
            model = AutoPeftModelForCausalLM.from_pretrained(checkpoint_path, **model_kwargs)
        except Exception as peft_error:
            print("PEFT load failed, falling back to AutoModelForCausalLM")
            print(peft_error)

    if model is None:
        model = AutoModelForCausalLM.from_pretrained(checkpoint_path, **model_kwargs)

    model.eval()
    return tok, model


def pick_dataset_question(dataset_path, split_name="test", index=0, random_pick=False, question_field="problem"):
    ds = load_from_disk(dataset_path)
    if hasattr(ds, "keys"):
        if split_name not in ds:
            raise ValueError(f"Split '{split_name}' not found in dataset")
        split_ds = ds[split_name]
    else:
        split_ds = ds

    if len(split_ds) == 0:
        raise ValueError("Selected split is empty")

    if random_pick:
        index = random.randint(0, len(split_ds) - 1)
    else:
        index = max(0, min(index, len(split_ds) - 1))

    row = split_ds[index]
    if question_field not in row:
        raise ValueError(f"Field '{question_field}' not found in dataset row. Available: {list(row.keys())}")

    question = str(row[question_field]).strip()
    expected = str(row.get("expected_answer", "")).strip()
    return question, expected, index, len(split_ds)


def build_prompt(system_prompt, question):
    return f"{system_prompt}\\n\\nProblem:\\n{question}\\n\\nSolution:\\n"


def run_generation(model, tok, prompt, max_new_tokens=256, temperature=0.7, top_p=0.95, do_sample=True):
    encoded = tok(prompt, return_tensors="pt")
    device = next(model.parameters()).device
    encoded = {k: v.to(device) for k, v in encoded.items()}

    with torch.no_grad():
        out = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            top_p=top_p if do_sample else None,
            eos_token_id=tok.eos_token_id,
            pad_token_id=tok.pad_token_id,
        )

    text = tok.decode(out[0], skip_special_tokens=True)
    return text

In [ ]:
# Demo controls
CHECKPOINT_PATH = "/content/drive/MyDrive/Math Olympiad Competition/Experimentation/models/sft_qwen_exp2/checkpoint-500"
TOKENIZER_PATH = None  # optional: set to a tokenizer/base model path if needed

# Choose where question comes from: "manual" or "dataset"
QUESTION_SOURCE = "manual"

# If QUESTION_SOURCE == "manual"
MANUAL_QUESTION = "Find the value of x if 2x + 5 = 17."

# If QUESTION_SOURCE == "dataset"
DATASET_PATH = "/content/drive/MyDrive/Math Olympiad Competition/Experimentation/processed_splits/omr_aimo3/splits/100000"
DATASET_SPLIT = "test"
DATASET_INDEX = 0
RANDOM_DATASET_QUESTION = False
QUESTION_FIELD = "problem"

# Prompt you want the model to follow
SYSTEM_PROMPT = "You are an expert olympiad math solver. Think step by step, then end with FINAL_ANSWER: <integer>."

# Generation settings
MAX_NEW_TOKENS = 512
DO_SAMPLE = True
TEMPERATURE = 0.7
TOP_P = 0.95

In [ ]:
print("Loading checkpoint:", CHECKPOINT_PATH)
tok, model = load_demo_model(CHECKPOINT_PATH, tokenizer_path=TOKENIZER_PATH)
print("Model loaded successfully")

In [ ]:
if QUESTION_SOURCE == "manual":
    question = MANUAL_QUESTION.strip()
    expected_answer = ""
    picked_index = None
    total_items = None
elif QUESTION_SOURCE == "dataset":
    question, expected_answer, picked_index, total_items = pick_dataset_question(
        dataset_path=DATASET_PATH,
        split_name=DATASET_SPLIT,
        index=DATASET_INDEX,
        random_pick=RANDOM_DATASET_QUESTION,
        question_field=QUESTION_FIELD,
    )
else:
    raise ValueError("QUESTION_SOURCE must be 'manual' or 'dataset'")

prompt = build_prompt(SYSTEM_PROMPT, question)

print("Question source:", QUESTION_SOURCE)
if QUESTION_SOURCE == "dataset":
    print(f"Picked dataset row: {picked_index} / {total_items - 1}")
    print("Expected answer:", expected_answer)
print("\nPrompt preview:\n")
print(prompt[:1200])

In [ ]:
if "extract_model_answer" not in globals():
    import re
    _final_re = re.compile(r"FINAL_ANSWER:\s*([+-]?\d+)\b")
    _last_int_re = re.compile(r"\b([+-]?\d+)\b")

    def extract_model_answer(text):
        if text is None:
            return None
        m = _final_re.findall(text)
        if m:
            return str(int(m[-1]))
        f = _last_int_re.findall(text)
        if f:
            return str(int(f[-1]))
        return None

response_text = run_generation(
    model=model,
    tok=tok,
    prompt=prompt,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    do_sample=DO_SAMPLE,
    )

predicted_answer = extract_model_answer(response_text)
expected_norm = str(int(expected_answer)) if str(expected_answer).strip().lstrip("+-").isdigit() else None

print("\n===== MODEL OUTPUT =====\n")
print(response_text)

print("\n===== ANSWER SUMMARY =====")
print("Predicted FINAL_ANSWER:", predicted_answer)
if QUESTION_SOURCE == "dataset":
    print("Expected answer:", expected_norm if expected_norm is not None else expected_answer)
    if predicted_answer is not None and expected_norm is not None:
        print("Correct:", predicted_answer == expected_norm)
    else:
        print("Correct: cannot determine (missing parsed predicted or expected answer)")
else:
    print("Expected answer: N/A (manual question mode)")